In [1]:
import ee
from pathlib import Path
import subprocess

ee.Initialize()


In [2]:
poa = ee.Geometry.Rectangle([-51.30344, -30.26945, -51.018852, -29.932474])
COLLECTION_ID = 'LANDSAT/LC08/C02/T1_L2'
START_DATE = '2018-01-01'
END_DATE = '2025-01-01'
EXPORT_FOLDER = 'earthengine'
EXPORT_SCALE = 30
MAX_PIXELS = 1e13


In [3]:
def mask_and_scale_lst(image):
    qa = image.select('QA_PIXEL')
    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
    )
    lst = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST_C').updateMask(mask)
    return lst.copyProperties(image, ['system:time_start'])

def add_month(image):
    return image.set('month', image.date().get('month'))

lst_col = (
    ee.ImageCollection(COLLECTION_ID)
    .filterBounds(poa)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq('PROCESSING_LEVEL', 'L2SP'))
    .map(mask_and_scale_lst)
    .map(add_month)
)

for m in range(1, 13):
    img = lst_col.filter(ee.Filter.eq('month', m)).mean().clip(poa).rename(f'month_{m:02d}_climatology_daytime_lst_c')
    name = f'poa_month_{m:02d}_climatology_daytime_lst_c'
    task = ee.batch.Export.image.toDrive(
        image=img, description=name, folder=EXPORT_FOLDER, fileNamePrefix=name,
        region=poa, scale=EXPORT_SCALE, crs='EPSG:3857', maxPixels=MAX_PIXELS, fileFormat='GeoTIFF'
    )
    task.start()
    print('Started Drive export:', name)


Started Drive export: poa_month_01_climatology_daytime_lst_c
Started Drive export: poa_month_02_climatology_daytime_lst_c
Started Drive export: poa_month_03_climatology_daytime_lst_c
Started Drive export: poa_month_04_climatology_daytime_lst_c
Started Drive export: poa_month_05_climatology_daytime_lst_c
Started Drive export: poa_month_06_climatology_daytime_lst_c
Started Drive export: poa_month_07_climatology_daytime_lst_c
Started Drive export: poa_month_08_climatology_daytime_lst_c
Started Drive export: poa_month_09_climatology_daytime_lst_c
Started Drive export: poa_month_10_climatology_daytime_lst_c
Started Drive export: poa_month_11_climatology_daytime_lst_c
Started Drive export: poa_month_12_climatology_daytime_lst_c


### Convert to COG and Generate Tiles (all 12 months)

In [4]:
base = Path('data')
colors_txt = base / 'monthly_climatology_colors.txt'
out_root = Path('out/monthly_climatology')

for m in range(1, 13):
    in_tif = base / f'poa_month_{m:02d}_climatology_daytime_lst_c.tif'
    out_dir = out_root / f'month_{m:02d}'
    cog_tif = out_dir / f'poa_month_{m:02d}_climatology_daytime_lst_c_cog.tif'
    visual_tiles_dir = out_dir / 'tiles_visual'
    value_tiles_dir = out_dir / 'tiles_values'
    out_dir.mkdir(parents=True, exist_ok=True)

    subprocess.run(['gdal_translate', str(in_tif), str(cog_tif), '-of', 'COG', '-ot', 'Float32', '-co', 'COMPRESS=DEFLATE', '-co', 'RESAMPLING=NEAREST', '-co', 'OVERVIEWS=AUTO'], check=True)
    colorized_tif = out_dir / f'poa_month_{m:02d}_climatology_daytime_lst_c_colorized.tif'
    subprocess.run(['gdaldem', 'color-relief', str(cog_tif), str(colors_txt), str(colorized_tif)], check=True)
    visual_tiles_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(['gdal2tiles.py', '-r', 'near', '-z', '8-15', '--xyz', '-w', 'none', str(colorized_tif), str(visual_tiles_dir)], check=True)

    value_encoded_tif = out_dir / f'poa_month_{m:02d}_climatology_daytime_lst_c_value_encoded_rgb.tif'
    base_expr = 'rint(clip((A*100)+10000,0,16777215)).astype(int64)'
    subprocess.run(['gdal_calc.py', '-A', str(cog_tif), '--calc', f'bitwise_and({base_expr},255)', '--calc', f'bitwise_and(right_shift({base_expr},8),255)', '--calc', f'bitwise_and(right_shift({base_expr},16),255)', '--type', 'Byte', '--NoDataValue', '0', '--overwrite', '--outfile', str(value_encoded_tif)], check=True)
    value_tiles_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(['gdal2tiles.py', '-r', 'near', '-z', '8-15', '--xyz', '-w', 'none', str(value_encoded_tif), str(value_tiles_dir)], check=True)
    print('Done month', m)

print('Decode value tiles with: lst_c = (R + 256*G + 65536*B - 10000) / 100')


Input file size is 1057, 1447
0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.


Warning 1: Input dataset has no nodata value. Ignoring 'nv' entry in color palette
Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0...10

Generating Overview Tiles:


...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.


...

.10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.


<string>:1: RuntimeWarning: invalid value encountered in cast
Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0...

Generating Overview Tiles:


10...20...30...40...50...60...70...80...90...100 - done.
Done month 12
Decode value tiles with: lst_c = (R + 256*G + 65536*B - 10000) / 100
